Läser in alla regiondata för rumsbeläggning och gästnätter. Gör lite feature engineering

In [43]:
import pandas as pd

In [44]:
from pathlib import Path

print(Path.cwd())

/Users/nicklas.thegerstrom/ws/ml/AppliedAI/tourism_weather


In [45]:
from pathlib import Path

DATA_DIR = Path("/Users/nicklas.thegerstrom/ws/ml/AppliedAI/tourism_weather/data")

In [46]:

def load_folder_simple(folder_path, value_name):
   
    dfs = []

    for file in Path(folder_path).glob("*.xlsx"):
        df = pd.read_excel(file, skiprows=4)
        df.columns = ["month_name", "year", value_name]
        df["region"] = file.stem
        dfs.append(df)

    return pd.concat(dfs, ignore_index=True)

In [47]:
guest_df = load_folder_simple(DATA_DIR / "guest_nights", "guest_nights")
occ_df = load_folder_simple(DATA_DIR / "occupancy", "occupancy_rate")

/opt/anaconda3/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/opt/anaconda3/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/opt/anaconda3/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/opt/anaconda3/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/opt/anaconda3/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook con

In [49]:
month_map = {
    "Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4,
    "Maj": 5, "Jun": 6, "Jul": 7, "Aug": 8,
    "Sep": 9, "Okt": 10, "Nov": 11, "Dec": 12
}

def add_date(df):
    df = df.copy()

    df["month"] = df["month_name"].map(month_map)
    df["year"] = pd.to_numeric(df["year"], errors="coerce")

    df = df.dropna(subset=["year", "month"])

    df["date"] = pd.to_datetime(
        dict(
            year=df["year"].astype(int),
            month=df["month"].astype(int),
            day=1
        )
    )

    return df

In [50]:
guest_df = add_date(guest_df)
occ_df = add_date(occ_df)

In [51]:
print(guest_df.columns)
print(occ_df.columns)

Index(['month_name', 'year', 'guest_nights', 'region', 'month', 'date'], dtype='object')
Index(['month_name', 'year', 'occupancy_rate', 'region', 'month', 'date'], dtype='object')


In [52]:
df_t= guest_df.merge(
    occ_df,
    on=["region", "date"],
    how="inner"
)

In [53]:
df_t.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1161 entries, 0 to 1160
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   month_name_x    1161 non-null   object        
 1   year_x          1161 non-null   float64       
 2   guest_nights    1161 non-null   object        
 3   region          1161 non-null   object        
 4   month_x         1161 non-null   float64       
 5   date            1161 non-null   datetime64[ns]
 6   month_name_y    1161 non-null   object        
 7   year_y          1161 non-null   float64       
 8   occupancy_rate  1161 non-null   object        
 9   month_y         1161 non-null   float64       
dtypes: datetime64[ns](1), float64(4), object(5)
memory usage: 90.8+ KB


In [54]:
df_t = df_t.rename(columns={
    "year_x": "year",
    "month_x": "month"
})

df_t = df_t.drop(columns=[
    "month_name_x",
    "month_name_y",
    "year_y",
    "month_y"
])

In [55]:
df_t["guest_nights"] = pd.to_numeric(df_t["guest_nights"], errors="coerce")
df_t["occupancy_rate"] = pd.to_numeric(df_t["occupancy_rate"], errors="coerce")

In [56]:
df_t = df_t.dropna().sort_values(["region", "date"]).reset_index(drop=True)

In [57]:
df_t.info()
df_t.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1161 entries, 0 to 1160
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   year            1161 non-null   float64       
 1   guest_nights    1161 non-null   int64         
 2   region          1161 non-null   object        
 3   month           1161 non-null   float64       
 4   date            1161 non-null   datetime64[ns]
 5   occupancy_rate  1161 non-null   float64       
dtypes: datetime64[ns](1), float64(3), int64(1), object(1)
memory usage: 54.6+ KB


,year,guest_nights,region,month,date,occupancy_rate
0,2015.0,815341,dalarna,7.0,2015-07-01,0.554058
1,2015.0,459539,dalarna,8.0,2015-08-01,0.427013
2,2015.0,172431,dalarna,9.0,2015-09-01,0.327624
3,2015.0,154394,dalarna,10.0,2015-10-01,0.285888
4,2015.0,127696,dalarna,11.0,2015-11-01,0.280259


In [58]:
df_t["year"] = df_t["year"].astype(int)
df_t["month"] = df_t["month"].astype(int)

In [59]:
df_t.groupby("region").size()

region
dalarna           129
gotland           129
jamtland          129
norrbotten        129
skane             129
stockholm         129
vasterbotten      129
vasternorrland    129
vastragotaland    129
dtype: int64

In [60]:
df_t["guest_lag1"] = df_t.groupby("region")["guest_nights"].shift(1)
df_t["guest_lag2"] = df_t.groupby("region")["guest_nights"].shift(2)
df_t["guest_lag12"] = df_t.groupby("region")["guest_nights"].shift(12)

df_t["occ_lag1"] = df_t.groupby("region")["occupancy_rate"].shift(1)

In [61]:
df_t[["region", "date", "guest_nights", "guest_lag1"]].head()

,region,date,guest_nights,guest_lag1
0,dalarna,2015-07-01,815341,NaN
1,dalarna,2015-08-01,459539,815341.0
2,dalarna,2015-09-01,172431,459539.0
3,dalarna,2015-10-01,154394,172431.0
4,dalarna,2015-11-01,127696,154394.0


In [62]:
df_t.isna().sum()

year                0
guest_nights        0
region              0
month               0
date                0
occupancy_rate      0
guest_lag1          9
guest_lag2         18
guest_lag12       108
occ_lag1            9
dtype: int64

In [63]:
import pandas as pd
import re
from pathlib import Path

def load_smhi_temp(path, region):
    rows = []

    with open(path, encoding="utf-8-sig") as f:
        for line in f:
            parts = line.strip().split(";")

            for i, value in enumerate(parts):
                value = value.strip()

                # hittar månad, t.ex. 2015-07
                if re.match(r"^\d{4}-\d{2}$", value):
                    if i + 1 < len(parts):
                        temp = parts[i + 1].replace(",", ".")
                        rows.append({
                            "region": region,
                            "date": pd.to_datetime(value),
                            "temp_mean": pd.to_numeric(temp, errors="coerce")
                        })

    weather = pd.DataFrame(rows)
    weather = weather.dropna().reset_index(drop=True)

    return weather

In [65]:
from pathlib import Path

weather_parts = []

for file in Path("data/smhi/temp").glob("*.csv"):
    region = file.stem
    weather_parts.append(load_smhi_temp(file, region))

weather_df = pd.concat(weather_parts, ignore_index=True)

In [66]:
weather_df["region"].unique()
weather_df.groupby("region").size()

region
dalarna           684
gotland           964
jamtland          983
lulea             973
skane             507
stockholm         351
sundsvall         939
umea              731
vastragotaland    432
dtype: int64

In [67]:
df_t["region"].unique()

array(['dalarna', 'gotland', 'jamtland', 'norrbotten', 'skane',
       'stockholm', 'vasterbotten', 'vasternorrland', 'vastragotaland'],
      dtype=object)

In [68]:
weather_df["region"] = weather_df["region"].replace({
    "lulea": "norrbotten",
    "umea": "vasterbotten",
    "sundsvall": "vasternorrland"
})

In [69]:
set(df_t["region"].unique()) - set(weather_df["region"].unique())

set()

In [70]:
df_final = df_t.merge(
    weather_df,
    on=["region", "date"],
    how="left"
)

In [71]:
df_final.groupby("region")["temp_mean"].apply(lambda x: x.isna().sum())

region
dalarna           2
gotland           2
jamtland          2
norrbotten        2
skane             7
stockholm         2
vasterbotten      2
vasternorrland    2
vastragotaland    4
Name: temp_mean, dtype: int64

In [72]:
df_model = df_final.dropna().reset_index(drop=True)

In [75]:
def load_smhi_daily_rain(path, region):
    rows = []

    with open(path, encoding="utf-8-sig") as f:
        for line in f:
            parts = [p.strip() for p in line.strip().split(";")]

            # SMHI regnfil:
            # 0 = från datetime
            # 1 = till datetime
            # 2 = representativt dygn
            # 3 = nederbörd
            if len(parts) >= 4:
                date = pd.to_datetime(parts[2], errors="coerce")
                rain = pd.to_numeric(parts[3].replace(",", "."), errors="coerce")

                if pd.notna(date) and pd.notna(rain):
                    rows.append({
                        "region": region,
                        "date": date,
                        "rain_mm": rain
                    })

    return pd.DataFrame(rows).reset_index(drop=True)

Aggregera regn från dag, till månad.

In [77]:
from pathlib import Path

rain_parts = []

for file in Path("data/smhi/regn").glob("*.csv"):
    region = file.stem
    rain_parts.append(load_smhi_daily_rain(file, region))

rain_df = pd.concat(rain_parts, ignore_index=True)

In [78]:
rain_df.head()
rain_df.groupby("region").size()

region
dalarna           60360
gotland           23427
jamtland          13825
norrbotten        25120
skane             11022
stockholm         26749
vasterbotten      19968
vasternorrland    19878
vastragotaland    16438
dtype: int64

In [79]:
rain_df["month"] = rain_df["date"].dt.to_period("M")

In [80]:
rain_df["month"] = rain_df["date"].dt.to_period("M")

rain_monthly = (
    rain_df
    .groupby(["region", "month"])
    .agg(
        rain_sum=("rain_mm", "sum"),
        rain_days=("rain_mm", lambda x: (x > 1).sum()),
        rain_mean=("rain_mm", "mean")
    )
    .reset_index()
)

rain_monthly["date"] = rain_monthly["month"].dt.to_timestamp()
rain_monthly = rain_monthly.drop(columns="month")

In [81]:
rain_monthly.head()

,region,rain_sum,rain_days,rain_mean,date
0,dalarna,98.4,12,3.174194,1860-01-01
1,dalarna,13.5,3,0.465517,1860-02-01
2,dalarna,27.1,5,0.874194,1860-03-01
3,dalarna,63.4,10,2.113333,1860-04-01
4,dalarna,55.3,8,1.783871,1860-05-01


In [82]:
rain_monthly = rain_monthly[
    (rain_monthly["date"] >= df_t["date"].min()) &
    (rain_monthly["date"] <= df_t["date"].max())
].copy()

In [83]:
rain_monthly.head()
rain_monthly.groupby("region").size()

region
dalarna           127
gotland           127
jamtland          127
norrbotten        127
skane             127
stockholm         127
vasterbotten      127
vasternorrland    127
vastragotaland    127
dtype: int64

In [84]:
rain_monthly.groupby("region")["date"].agg(["min", "max"])

,min,max
region,,
dalarna,2015-07-01,2026-01-01
gotland,2015-07-01,2026-01-01
jamtland,2015-07-01,2026-01-01
norrbotten,2015-07-01,2026-01-01
skane,2015-07-01,2026-01-01
stockholm,2015-07-01,2026-01-01
vasterbotten,2015-07-01,2026-01-01
vasternorrland,2015-07-01,2026-01-01
vastragotaland,2015-07-01,2026-01-01


In [85]:
df_final = df_final.merge(
    rain_monthly,
    on=["region", "date"],
    how="left"
)

In [86]:
df_final.groupby("region")[["rain_sum", "rain_days"]].apply(lambda x: x.isna().sum())

,rain_sum,rain_days
region,,
dalarna,2,2
gotland,2,2
jamtland,2,2
norrbotten,2,2
skane,2,2
stockholm,2,2
vasterbotten,2,2
vasternorrland,2,2
vastragotaland,2,2


In [87]:
df_model = df_final.dropna().reset_index(drop=True)

In [88]:
df_model.sample(20)

,year,guest_nights,region,month,date,occupancy_rate,guest_lag1,guest_lag2,guest_lag12,occ_lag1,temp_mean,rain_sum,rain_days,rain_mean
933,2018,435153,vastragotaland,1,2018-01-01,0.487301,450160.0,546780.0,421362.0,0.484198,0.1,104.2,14.0,3.361290
461,2016,869083,skane,8,2016-08-01,0.727004,1222152.0,624678.0,847942.0,0.764457,16.1,62.9,11.0,2.029032
286,2021,467351,jamtland,3,2021-03-01,0.489995,405955.0,314559.0,352855.0,0.438469,-0.6,15.8,6.0,0.509677
235,2016,267248,jamtland,12,2016-12-01,0.407133,104872.0,83352.0,292303.0,0.403084,-0.6,20.4,9.0,0.658065
851,2020,86642,vasternorrland,10,2020-10-01,0.499524,83003.0,148278.0,83819.0,0.489069,6.2,104.9,15.0,3.383871
732,2020,103196,vasterbotten,6,2020-06-01,0.271336,56266.0,45315.0,192235.0,0.187617,16.1,77.8,5.0,2.593333
44,2020,541896,dalarna,3,2020-03-01,0.382623,752758.0,648623.0,666322.0,0.588193,1.8,24.6,5.0,0.793548
204,2023,17276,gotland,12,2023-12-01,0.231131,18907.0,41752.0,18759.0,0.260974,1.2,52.2,14.0,1.683871
259,2018,267595,jamtland,12,2018-12-01,0.457875,95128.0,81113.0,285816.0,0.376713,-3.7,70.8,12.0,2.283871
247,2017,285816,jamtland,12,2017-12-01,0.440245,111514.0,93934.0,267248.0,0.446340,-4.3,62.0,11.0,2.000000


FÄRDIG!!!

In [89]:
df_model.to_csv("sweden_tourism_weather.csv", index=False)